In [0]:
# %iverpip install openai>=1.56.0
# %pip install azure-keyvault-secrets azure-identity
# %restart_python

In [0]:
from openai import AsyncAzureOpenAI
import asyncio
import mlflow
import json
import re
import pandas as pd
import ast
import numpy as np
import os
from datetime import datetime
from copy import deepcopy
from scipy.stats import entropy, mode
from sklearn.metrics import f1_score
from tqdm.asyncio import tqdm
from sklearn.metrics import f1_score
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

mlflow.openai.autolog(disable=True)
mlflow.autolog(disable=True)

# allows asyncio inside main
import nest_asyncio
nest_asyncio.apply()

In [0]:
# # Primary endpoint
API_KEY = 
client = AsyncAzureOpenAI(
                api_version="2024-10-21",
                azure_endpoint="https://oai-rtai-dev-aoai-east-001.openai.azure.com/",
                api_key=API_KEY,
            )



In [0]:
def get_system_prompt() -> str:
    """Reads and returns the system prompt from prompt.md."""
    try:
        with open("./prompts/PCS_autoprompt.md", "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return "Error: prompt.md not found. Please ensure the file exists in the correct directory."

#TODO update for PCS
def get_schema() -> dict:
    schema_definition = {
        "name": "agent_performance_analysis",
        "description": "Analyzes Spectrum call transcripts to classify the performance of the Sprectrum Customer Solutions Agent.",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "class": {
                    "type": "string",
                    "enum": [
                        "Negative",
                        "Neutral",
                        "Positive",
                    ],
                    "description": "The most appropriate rating of the agent performance."
                },
                "rationale": {
                    "type": "string",
                    "description": "A thorough explanation of reasoning." 
                },
                "citations": {
                    "type": "array",
                    "items": {
                        "type": "integer"
                    },
                    "description": "List of line numbers from the transcript that served as primary evidence for the classification."
                },
                "score": {
                    "type": "integer",
                    "minimum": 0,
                    "maximum": 10,
                    "description": "A numeric score from 0 to 10 representing the agent's performance."
                }
            },
            "required": ["class", "rationale", "citations", "score"],
            "additionalProperties": False
        }
    }
   
    return {
        "type": "json_schema",
        "json_schema": schema_definition
    }

In [0]:
# LLM class
async def execute_completion(config: dict, 
                             user_input: str, 
                             client: AsyncAzureOpenAI = client):
    response_format = config["response_format"]
    system_prompt = config["system_prompt"]
    reasoning_effort = config["reasoning_effort"]
    model_name = config["model_name"]

    n_retries = config['n_retries']
    for attempt in range(n_retries):
        try:
            response = await client.chat.completions.with_raw_response.create(
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_input}
                ],
                response_format=response_format,
                reasoning_effort=reasoning_effort,
                model=model_name,
                timeout=60.0
            )
            headers = response.headers
            parsed = response.parse()
            content = parsed.choices[0].message.content

            # capture debug info
            debug_info = str({
                "rate_limits": {
                    "tokens_left": headers.get('x-ratelimit-remaining-tokens'),
                    "requests_left": headers.get('x-ratelimit-remaining-requests'),
                },
                "usage": {
                    "prompt_tokens": parsed.usage.prompt_tokens,
                    "completion_tokens": parsed.usage.completion_tokens,
                    "cached_tokens": getattr(parsed.usage.prompt_tokens_details, 'cached_tokens', 0),
                    "total_tokens": parsed.usage.total_tokens,
                },
                "metadata": {
                    "model": parsed.model,
                    "finish_reason": parsed.choices[0].finish_reason,
                    "request_id": headers.get('x-request-id'),
                    "processing_ms": headers.get('openai-processing-ms')
                }
            })

            return content, debug_info
        
        except Exception as e:
            print(f"Error occurred: {e}")
            fallback_content = json.dumps({
                "class": "error",
                "certainty": "error",
                "rationale": "error",
                "citations": ["error"]
            })
            fallback_debug_info = str({
                "rate_limits": {"tokens_left": 0, "requests_left": 0},
                "usage": {"prompt_tokens": 0, "completion_tokens": 0, "cached_tokens": 0, "total_tokens": 0},
                "metadata": {"model": 0, "finish_reason": 0, "request_id": 0, "processing_ms": 0}
            })
            return fallback_content, fallback_debug_info

In [0]:

async def async_driver_semaphore(config: dict,
                 d: str,
                 sem: asyncio.Semaphore = None,
                 client: AsyncAzureOpenAI = client) -> pd.DataFrame:
    
    async with sem:
        return await execute_completion(config, d, client)

async def async_driver(config: dict,
                 df: pd.DataFrame, 
                 client: AsyncAzureOpenAI = client) -> pd.DataFrame:
    
    # cast our data to a list
    data = df['Text'].tolist()

    # initialize semaphore
    sem = asyncio.Semaphore(config['semaphore_value'])

    # make the task list
    tasks = [
        async_driver_semaphore(config, d, sem, client)
        for d in data
    ]

    # unpack the list when passing to 
    results = await tqdm.gather(*tasks)
    completions, debug = zip(*results) 

    # enrich df with responses
    df['predictions'] = list(completions)
    df['debug_info'] = list(debug)

    return df

def extract_token_data(df):
   
    # 1. Convert the 'debug_info' strings back into real dictionaries
    df['debug_dict'] = df['debug_info'].apply(ast.literal_eval)

    # 2. Extract the specific usage values
    df['prompt_tokens'] = df['debug_dict'].apply(lambda x: x['usage']['prompt_tokens'])
    df['cached_tokens'] = df['debug_dict'].apply(lambda x: x['usage']['cached_tokens'])
    df['completion_tokens'] = df['debug_dict'].apply(lambda x: x['usage']['completion_tokens'])
    df['total_tokens'] = df['debug_dict'].apply(lambda x: x['usage']['total_tokens'])

    # 3. Calculate cache rate per row
    df['cache_rate'] = df['cached_tokens'] / df['prompt_tokens']

    # 4. Get the final averages
    avg_prompt_tokens = df['prompt_tokens'].mean()
    avg_cache_rate = df['cache_rate'].mean()
    avg_completion_tokens = df['completion_tokens'].mean()
    avg_total_tokens = df['total_tokens'].mean()

    # print(f"Avg Tokens per Row: {avg_prompt_tokens:.2f}")
    # print(f"Avg Prompt Cache Rate: {avg_cache_rate:.2%}")
    # print(f"Avg Completion Tokens per Row: {avg_completion_tokens:.2f}")
    # print(f"Avg Total Tokens per Row: {avg_total_tokens:.2f}")

    return df

In [0]:
def extract_data(df):
    parsed = df['predictions'].apply(json.loads)

    df['class'] = parsed.apply(lambda x: x.get('class'))
    df['score'] = parsed.apply(lambda x: x.get('score'))
    df['rationale'] = parsed.apply(lambda x: x.get('rationale'))
    df['citations'] = parsed.apply(lambda x: x.get('citations'))

    # drop classes not included in original analysis
    excluded_labels = ['']
    try:
        df = df[~df['label'].isin(excluded_labels)].copy()
    except Exception as e:
        print(f"Error filtering excluded_labels: {e}")

    # Standardize label values using mapping dictionary
    mapping = {
    }

    try:
        df['label'] = df['label'].apply(lambda val: mapping.get(val, val))
    except Exception as e:
        print(f"Error mapping label values: {e}")

    try:        
        df = extract_token_data(df)
    except Exception as e:
        print(f'Error in extract_token_data: {e}')

    return df

In [0]:
def eval(df_original, config):
    # WARNING: Subtracting 2 from all model predictions before scoring
    df_run = asyncio.run(async_driver(config, df_original.copy(), client))
    df_run = extract_data(df_run)
    df_run = df_run[df_run["label"] != "nan"].copy()

    df_run["score"] = pd.to_numeric(df_run["score"], errors="coerce")
    df_run["label"] = pd.to_numeric(df_run["label"], errors="coerce")

    # Subtract 2 from all model predictions
    df_run["score"] = df_run["score"] #- 2

    df_run = df_run.dropna(subset=["score", "label"])

    df_run["correct"] = ((df_run["score"] - df_run["label"]).abs() <= 1.5).astype(int)

    overall_accuracy = df_run["correct"].mean()
    print(f"Overall accuracy: {overall_accuracy:.3f}")

    eval_df = (
        df_run.groupby("label")
        .agg(
            label_count=("label", "count"),
            correct_count=("correct", "sum")
        )
        .reset_index()
    )

    eval_df["accuracy"] = eval_df["correct_count"] / eval_df["label_count"]

    return df_run, eval_df

In [0]:
# echo but can add modifications to this function if desired
def enrich_text(df):
    return df

In [0]:
# load the data 
#internally expected Text and label
def load_data(path: str, config: dict = None) -> pd.DataFrame:
    df = pd.read_csv(path)
    df['Text'] = df['Text'].str.replace('/n', '', regex=False)

    try:
        df['label'] = df['Consensus_Label']
    except:
        print("No column \"Consensus_Label\" found")

    # apply enrichment to text
    df = enrich_text(df)

    return df.astype(str)

In [0]:
config = {
        "response_format": get_schema(),
        "system_prompt": get_system_prompt(),
        "reasoning_effort": "low",
        "model_name": "gpt-5-mini",
        "semaphore_value": 10,
        "n_retries": 30,
        "eval": True,
    }

In [0]:
def LLM_driver(config = None):

    # Create a unique output directory based on date and time
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"./outputs/{timestamp}"
    os.makedirs(output_dir, exist_ok=True)

    df_original = load_data('./data/PCS_prepped2.csv', config)

    if config['eval']:
        df_run, eval_df = eval(df_original.copy(), config)
        
        print("First Run Outputs")
        df_run.to_csv(f"{output_dir}/df_run_0.csv", index=False)
    
        print("Individual Run Data")
        if not eval_df.empty:
            display(eval_df)
        eval_df.to_csv(f"{output_dir}/df_eval.csv", index=False)

        return df_run, eval_df

    else:
        df_run = asyncio.run(async_driver(config, df_original.copy(), client))
        df_run.to_csv(f"{output_dir}/df_run_0.csv", index=False)
        df = extract_data(df_run)
        #display(df)

        # Return empty DataFrame for eval output
        empty_df = pd.DataFrame()
        return df, empty_df

In [0]:
def LLM_driver1(config: dict, data_path: str):
    # """
    # Inference-only driver for unlabeled call center data.
    # Expects a CSV with at minimum: 'agentrecordingsessionid' and 'Text' columns.
    # Returns a DataFrame with the structured LLM output fields:
    #   - class (Negative/Neutral/Positive)
    #   - score (0-10)
    #   - rationale
    #   - citations
    # """

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"./outputs/{timestamp}"
    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(data_path)
    cols = df.columns
    required_cols = {'AgentRecordingSessionID', 'Text'}
    missing = required_cols - set(cols)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.dropna(subset=['Text'])
    df['Text'] = df['Text'].astype(str).str.replace('/n', '', regex=False)
    df = df[~df['Text'].isin(['nan', 'None', ''])].reset_index(drop=True)
    print(f"Records with valid text: {len(df)}")

    df = enrich_text(df)

    df_run = asyncio.run(async_driver(config, df.copy(), client))

    parsed = df_run['predictions'].apply(json.loads)
    df_run['class'] = parsed.apply(lambda x: x.get('class'))
    df_run['score'] = parsed.apply(lambda x: x.get('score'))
    df_run["score"] = df_run["score"] # - 2
    df_run['rationale'] = parsed.apply(lambda x: x.get('rationale'))
    df_run['citations'] = parsed.apply(lambda x: x.get('citations'))

    try:
        df_run = extract_token_data(df_run)
    except Exception as e:
        print(f"Error in extract_token_data: {e}")

    return df_run

In [0]:
df_results = LLM_driver1(config, './data/100_transcripts_0626.csv')
df_results.to_csv('./outputs/temp_run.csv', index=False)

Records with valid text: 101


100%|██████████| 101/101 [00:45<00:00,  2.23it/s]


In [0]:
df_results=pd.read_csv('./outputs/temp_run.csv')
# df_results['class'].value_counts()
df_results['score'].value_counts().to_csv('./data/score_counts.csv')


In [0]:
df_results_class = df_results.copy()

df_results_class['score_new'] = df_results_class['score']+2

# for score in df_results_class['score']:
#     score = score + 2

def map_score_to_class(entry):
    try:
        entry = float(entry)
        if entry >= 0 and entry <= 3:
            return 'Negative'
        elif entry >= 4 and entry <= 6:
            return 'Neutral'
        elif entry >= 7 and entry <= 10:
            return 'Positive'
        else:
            return 'Unknown'
    except:
        return 'Unknown'
    

df_results_class['class'] = df_results_class['score_new'].apply(map_score_to_class)

df_results_class['score_new'].value_counts()


score_new
6    3804
7     884
5     417
8     141
4      91
3      14
9       5
2       2
Name: count, dtype: int64

In [0]:
df_results_class['new_class'] = df_results_class['score_new'].apply(lambda x: 'Positive' if x == 9 else ('Negative' if 1 <= x <= 4 else ('Neutral' if 5 <= x <= 8 else None)))
# display(df_results_class)

In [0]:
df_results_class['new_class'].value_counts()

new_class
Neutral     5246
Negative     107
Positive       5
Name: count, dtype: int64

In [0]:
df_results_class.to_csv('./outputs/acc_management_5k_new_class.csv', index=False)

In [0]:
df_test = pd.read_csv('./data/acc_management_5k_new_class.csv')

/home/spark-d79b1d02-2192-45b3-93ee-06/.ipykernel/1987/command-7638249008721832-797334132:1: DtypeWarning: Columns (214,218,222,223,224,225,226,227,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,381,382,383,385,386,387,394) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('./data/acc_management_5k_new_class.csv')


In [0]:
df_test.head(2)

,Unnamed: 0,AgentRecordingSessionID,Text,Consensus_Label,predictions,debug_info,class,score,rationale,citations,debug_dict,prompt_tokens,cached_tokens,completion_tokens,total_tokens,cache_rate,score_new,new_class,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39,...,Unnamed: 355,Unnamed: 356,Unnamed: 357,Unnamed: 358,Unnamed: 359,Unnamed: 360,Unnamed: 361,Unnamed: 362,Unnamed: 363,Unnamed: 364,Unnamed: 365,Unnamed: 366,Unnamed: 367,Unnamed: 368,Unnamed: 369,Unnamed: 370,Unnamed: 371,Unnamed: 372,Unnamed: 373,Unnamed: 374,Unnamed: 375,Unnamed: 376,Unnamed: 377,Unnamed: 378,Unnamed: 379,Unnamed: 380,Unnamed: 381,Unnamed: 382,Unnamed: 383,Unnamed: 384,Unnamed: 385,Unnamed: 386,Unnamed: 387,Unnamed: 388,Unnamed: 389,Unnamed: 390,Unnamed: 391,Unnamed: 392,Unnamed: 393,Unnamed: 394
0,0,00016295C71547439091912CDF9DA93D,1 Agent: It was Michael. I'm a specialist here...,NaN,"{\n ""class"": ""Positive"",\n ""rationale"": ""The...","{'rate_limits': {'tokens_left': '3995109', 're...",Positive,6,The caller's core reason for calling was to re...,"[1, 3, 6, 22, 24, 29, 31, 41, 44, 46, 59, 67, ...","{'rate_limits': {'tokens_left': '3995109', 're...",4891,2560,680,5571,0.5234103455326109,8,Neutral,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,000ED3FC36913344A842A6C777F41516,1 Agent: Thank you for calling Spectrum. I Hop...,NaN,"{\n ""class"": ""Positive"",\n ""rationale"": ""Cus...","{'rate_limits': {'tokens_left': '3996320', 're...",Positive,6,Customer's core reason for calling: billing ap...,"[2, 3, 4, 5, 20, 32, 37, 41, 43, 49, 1, 7, 11,...","{'rate_limits': {'tokens_left': '3996320', 're...",3680,2560,607,4287,0.6956521739130435,8,Neutral,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [0]:
df_results['class'].value_counts()

class
Positive    4519
Neutral      777
Negative      62
Name: count, dtype: int64

In [0]:
df_results.to_csv('./outputs/temp_results.csv', index=False)

In [0]:
df_run, df_eval = LLM_driver(config)
df_run.to_csv('./outputs/temp_run.csv', index=False)
#df_eval.to_csv('./outputs/temp_eval.csv', index=False)
len(df_run)

100%|██████████| 38/38 [00:30<00:00,  1.25it/s]


Overall accuracy: 0.816
First Run Outputs
Individual Run Data


label,label_count,correct_count,accuracy
3,1,1,1.0
5,3,3,1.0
6,22,18,0.8181818181818182
7,8,8,1.0
8,4,1,0.25


38

In [0]:
display(df_run)

In [0]:
print("Label set:", set(df_run['label'].unique()))
print("Class set:", set(df_run['class'].unique()))
print(set(df_run['class'].unique()).issubset(set(df_run['label'].unique())))

# Numerical table of class distribution
class_counts = df_run['label'].value_counts().reset_index()
class_counts.columns = ['label', 'count']
display(class_counts)

Label set: {np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)}
Class set: {'Negative', 'Neutral', 'Positive'}
False


label,count
6,68
5,21
7,12
8,2
3,2
2,1
4,1
